In [2]:
# load the ImageNet dataset from torchvision
from torchvision import datasets,transforms, models
from torch.utils.data import DataLoader

# import torchlightning
import pytorch_lightning as pl

# set the current working directory to directory of this notebook
import os
os.chdir(os.path.dirname(os.path.abspath('__file__')))

In [1]:
# use pytorch lightning to load the dataset
from pytorch_lightning.utilities.types import EVAL_DATALOADERS
from torch.utils.data import DataLoader
from torch.utils.data import random_split
import torch.nn.functional as F

import torch
import pytorch_lightning as pl


class CIFAR100DataModule(pl.LightningDataModule):
    def __init__(self, data_path, batch_size=32, num_workers=8):
        super().__init__()
        self.data_path = data_path
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.transform = transforms.Compose([
                    transforms.ToTensor(),
                    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),  # Global Contrast Normalization
        ])

    def prepare_data(self):
        datasets.CIFAR100(self.data_path, train=True, download=True)
        datasets.CIFAR100(self.data_path, train=False, download=True)

    def setup(self, stage=None):
        if stage == 'fit' or stage is None:
            dataset = datasets.CIFAR100(self.data_path, train=True, transform=self.transform)

            # Split the dataset into training and validation set
            self.cifar100_train, self.cifar100_val = random_split(dataset, [45000, 5000])
            self.cifar100_test = datasets.CIFAR100(self.data_path, train=False, transform=self.transform)

    def train_dataloader(self):
        return DataLoader(self.cifar100_train, batch_size=self.batch_size, persistent_workers=True, num_workers=self.num_workers, shuffle=True)
    
    def test_dataloader(self):
        return DataLoader(self.cifar100_test, batch_size=self.batch_size, num_workers=self.num_workers, )

    def val_dataloader(self):
        return DataLoader(self.cifar100_val, batch_size=self.batch_size, persistent_workers=True, num_workers=self.num_workers)

In [3]:
## logger details: https://lightning.ai/docs/pytorch/stable/extensions/logging.html


import torch.nn.functional as F

class TransferLearning(pl.LightningModule):
    def __init__(self):
        super(TransferLearning, self).__init__()
        self.model = models.vgg11(weights=False)
        
    def forward(self, x):
        return self.model(x)
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self.model(x)
        loss = F.cross_entropy(y_hat, y)
        self.log('train_loss', loss, on_epoch=True, prog_bar=True)

        # determine the accuracy
        _, y_pred = torch.max(y_hat, dim=1)
        acc = torch.tensor(torch.sum(y_pred == y).item() / len(y), dtype=torch.float32)
        self.log('train_acc', acc * 100)

        return loss
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-4)
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self.model(x)
        loss = F.cross_entropy(y_hat, y)
        self.log('val_loss', loss)
    
# add a custom layer
#model.classifier[6] = nn.Linear(4096, 2)

model = TransferLearning()
dm = CIFAR100DataModule(data_path='datasets')
trainer = pl.Trainer(max_epochs=10)
trainer.fit(model, dm)

e:\miniconda3\envs\chaos\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
e:\miniconda3\envs\chaos\Lib\site-packages\pytorch_lightning\trainer\connectors\logger_connector\logger_connector.py:75: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default


Files already downloaded and verified
Files already downloaded and verified


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type | Params | Mode 
---------------------------------------
0 | model | VGG  | 132 M  | train
---------------------------------------
132 M     Trainable params
0         Non-trainable params
132 M     Total params
531.453   Total estimated model params size (MB)
32        Modules in train mode
0         Modules in eval mode


Epoch 0: 100%|██████████| 1407/1407 [01:02<00:00, 22.68it/s, v_num=40, train_loss_step=4.260, train_loss_epoch=4.170]

RuntimeError: [enforce fail at inline_container.cc:603] . unexpected pos 1422601472 vs 1011559568